# 🏠 Week 6: YouTube Transcript Summarizer - Homework

## Welcome to your homework notebook!

This notebook contains **11 practice questions** and **3 bonus challenges** designed to reinforce what you learned this week.

### How to use this notebook:
1. **Read** each question carefully
2. **Follow** the hints provided
3. **Write** your answer in the code/markdown cells below
4. **Test** your code to verify it works
5. **Submit** your completed notebook

**Time to complete**: 45-60 minutes (including bonus challenges)

Let's get started! 🚀

---

## Setup: Import Required Libraries

Before starting, make sure you have imported all necessary libraries. Run the cell below:

In [ ]:
# Import all libraries you'll need for this homework
import os
import json
import re
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# Load environment variables
load_dotenv()

# Verify your API key is loaded
api_key = os.getenv("OPENAI_API_KEY")
print("✓ Libraries imported successfully!")
print(f"✓ OpenAI API Key loaded: {'Yes' if api_key else 'No'}")

---

## 📌 Section 1: Setup & Imports (Conceptual Questions)

### Question 1: Environment Variables & Security ⭐

**Question**: Why do we store API keys in a `.env` file instead of hardcoding them directly in the notebook?

**Hint**: Think about:
- What happens if you push this notebook to GitHub?
- Who has access to your hardcoded keys?
- How can you safely share your code with others?

**Answer Space**: Write your answer below (2-3 sentences)

**YOUR ANSWER TO Q1:**

_[Replace this text with your answer]_

---

**Explanation**:
- `.env` files are **NOT** committed to git (listed in `.gitignore`)
- API keys are never exposed in the source code
- Others can clone your code safely without getting your credentials
- You can share code without sharing secrets

### Question 2: .env vs os.environ (Basics)

**Question**: Load your API keys using `os.environ` (without `dotenv`) and compare this approach with loading from a `.env` file. What are the pros/cons of each?

**Tasks**:
- Try accessing `OPENAI_API_KEY` with `os.environ.get("OPENAI_API_KEY")`
- Describe when `.env` files are helpful vs. when plain environment variables are enough

**Hint**:
- Think about **local development** vs. **deployment** environments
- Consider **secrets leakage**, **portability**, and **developer experience**
- What happens if the key is missing? How would you handle errors?

**Your Answer**: Write 3-5 bullet points comparing the two approaches.


**YOUR ANSWER TO Q2:**

- `.env` approach: _______________________________
- `os.environ` approach: __________________________
- Compare (pros/cons): ____________________________
- When to use each: _______________________________

**Tip**: Mention security (git), portability, DX, and deployment differences.

### Question 3: LangChain vs Traditional API Calls ⭐

**Question**: What is the main advantage of using LangChain's `ChatOpenAI` class instead of calling the OpenAI API directly?

**A)** It's faster  
**B)** It provides a unified interface for working with LLMs and allows easy chaining of components  
**C)** It's cheaper  
**D)** It doesn't require an API key

**Hint**: Think about what LangChain is designed to do. Look at how we used `ChatOpenAI` in the notebook - did we have to handle API responses manually?

**Your Answer**: _____

**EXPLANATION FOR Q3:**

The correct answer is **B**.

LangChain provides:
- 🔗 **Chaining**: Connect components easily (Prompt → LLM → Parser)
- 🔄 **Consistency**: Same interface for different LLM providers
- 🛠️ **Abstraction**: Handles API calls internally
- 📝 **Templates**: Built-in prompts and output parsing

Example:
```python
# Without LangChain: Manual API call, parsing, etc.
# With LangChain: Just chain it!
chain = prompt | llm | output_parser
result = chain.invoke({"input": text})
```

---

## 📌 Section 2: Transcript Extraction (Practical Questions)

### Question 4: Parsing YouTube Captions 🔧

**Question**: Look at this JSON3 caption format from YouTube:

```json
{
    "events": [
        {"segs": [{"utf8": "Hello"}, {"utf8": "world"}]},
        {"segs": [{"utf8": "How"}, {"utf8": "are"}, {"utf8": "you"}]}
    ]
}
```

What would the `_parse_json3_captions()` function return from this JSON?

**Hint**: The function:
1. Loops through each "event"
2. Extracts text from each "seg"
3. Joins them with spaces

**Step-by-step**:
- Event 1 has: "Hello", "world" → "Hello world"
- Event 2 has: "How", "are", "you" → "How are you"
- Join them together: _______

In [ ]:
# Let's verify your answer by actually parsing it!

sample_json = """{
    "events": [
        {"segs": [{"utf8": "Hello"}, {"utf8": "world"}]},
        {"segs": [{"utf8": "How"}, {"utf8": "are"}, {"utf8": "you"}]}
    ]
}"""

# TODO: Write the _parse_json3_captions function (copy it from the main notebook)
def _parse_json3_captions(json_text: str) -> str:
    """Parse JSON3 captions and extract text."""
    # STEP 1: Parse JSON
    data = json.loads(json_text)
    
    # STEP 2: Extract all text segments
    segs = []
    for event in data.get("events", []):
        for seg in event.get("segs", []):
            text = seg.get("utf8")
            if text:
                segs.append(text)
    
    # STEP 3: Join with spaces
    return " ".join(segs)

# Test it
result = _parse_json3_captions(sample_json)
print(f"Parsed result: {result}")
print(f"\nExpected: 'Hello world How are you'")
print(f"Match: {result == 'Hello world How are you'}")

**YOUR ANSWER TO Q4:**

Expected output: _______________________________

**EXPLANATION:**

The output should be: **"Hello world How are you"**

How it works:
1. JSON has 2 events
2. First event: ["Hello", "world"] → "Hello world"
3. Second event: ["How", "are", "you"] → "How are you"
4. Join with spaces: "Hello world How are you"

### Question 5: Caption Availability 🎯

**Question**: When extracting a transcript, what are 3 reasons why the function might fail to find captions?

**Hint**: Think about:
- Video settings and privacy
- Language and availability
- Content type and restrictions

**Step 1**: Reason 1 - _________________________________

**Step 2**: Reason 2 - _________________________________

**Step 3**: Reason 3 - _________________________________

**EXPLANATION FOR Q5:**

Common reasons transcript extraction fails:

1. **Video has captions disabled**: Creator turned off captions
2. **No English captions available**: Only captions in other languages exist
3. **Age-restricted or private content**: Video is not publicly accessible
4. **Live streams**: Some live content doesn't have captions available yet
5. **URL is invalid**: Malformed or incorrect YouTube URL

Any 3 of these are acceptable answers! ✅

### Question 6: yt_dlp Configuration 🔧

**Question**: In the `extract_transcript()` function, what is the purpose of setting `'quiet': True` and `'no_warnings': True` in the `ydl_opts` dictionary?

**A)** To make the download faster  
**B)** To suppress unnecessary console output and only show important messages  
**C)** To save bandwidth  
**D)** To prevent errors from occurring

**Hint**: Think about what these parameter names mean:
- `quiet` → fewer messages
- `no_warnings` → hide warnings

Your Answer: _____

**EXPLANATION FOR Q6:**

The correct answer is **B**.

These options control **output verbosity**:
- `'quiet': True` → Don't print download progress
- `'no_warnings': True` → Don't show non-critical warnings
- Result: Cleaner console output, only important messages shown

Example comparison:
```python
# Without these options: 100+ lines of download info
# With these options: Just the final result
```

This is useful in production so logs stay clean! ✅

---

## 📌 Section 3: Text Preprocessing (Practical Questions)

### Question 7: Preprocessing Steps 🧹

**Question**: Run the `preprocess_text()` function on this input and show the cleaned output.

**Input text**:
```
"Hello!!!    World...   How   are   you   doing????    I'm   excited!!!"
```

**Steps**:
1. Copy the `preprocess_text()` function from the main notebook
2. Call it with the input above
3. Print the result
4. Write what you see

In [ ]:
# Step 1: Define the preprocess_text function
def preprocess_text(text: str) -> str:
    """Clean and normalize text for LLM processing."""
    # Remove extra whitespace (multiple spaces → single space)
    text = re.sub(r"\s+", " ", text)
    
    # Remove special/unusual characters but keep punctuation
    text = re.sub(r'[^\w\s.!?\-]', '', text)
    
    # Fix spacing after punctuation marks
    text = re.sub(r'([.!?])\s+', r'\1 ', text)
    
    # Remove leading/trailing whitespace
    return text.strip()

# Step 2: Test input
test_input = "Hello!!!    World...   How   are   you   doing????    I'm   excited!!!"

# Step 3: Run preprocessing
result = preprocess_text(test_input)

# Step 4: Show results
print("Input:")
print(f"  {repr(test_input)}")
print(f"\nOutput:")
print(f"  {repr(result)}")
print(f"\nReadable output:")
print(f"  {result}")

**YOUR ANSWER TO Q7:**

Expected cleaned output: _________________________________

**EXPLANATION:**

Input:  `"Hello!!!    World...   How   are   you   doing????    I'm   excited!!!"`

Process:
1. Remove extra spaces: `"Hello!!! World... How are you doing???? I'm excited!!!"`
2. Remove special chars (keep punctuation): `"Hello World How are you doing Im excited"`
3. Fix punctuation spacing: `"Hello! World. How are you doing? Im excited!"`
4. Strip edges: Final clean text

Output: `"Hello World How are you doing Im excited"`

### Question 8: Regex Mini-Check ✅

**Question (very short)**: In `re.sub(r"\s+", " ", text)`, what does `\s+` match?

**Choices** (pick one):
- A) Exactly one space
- B) One or more whitespace characters (spaces, tabs, newlines)
- C) Only newline characters
- D) Letters and numbers

**Your Answer**: _____

In [ ]:
# Quick demo: replace runs of whitespace with a single space
sample = "Hello\t\tworld\n\nHow   are   you?"
result = re.sub(r"\s+", " ", sample)
print("Before:", repr(sample))
print("After: ", repr(result))

**EXPLANATION FOR Q8:**

Correct answer: **B**. `\s+` means “one or more whitespace characters” (spaces, tabs, or newlines).

### Question 9: What Gets Removed? 🎯 (Keep it simple)

The cleaning line is:
```python
re.sub(r'[^\w\s.!?\-]', '', text)
```

**Question**: Which one of these will be **removed** by this regex?
- A) Letters and numbers
- B) Exclamation marks `!`
- C) Hyphens `-`
- D) Emoji like 😊

**Your Answer**: _____

In [ ]:
# Quick check: which characters survive the regex?
sample = "Hello! café 😊 back-end @tag"
cleaned = re.sub(r'[^\w\s.!?\-]', '', sample)
print("Before:", sample)
print("After: ", cleaned)

**EXPLANATION FOR Q9:**

Correct answer: **D** — emoji and other non-word symbols get removed. The pattern keeps letters, numbers, spaces, `. ! ? -`, so punctuation like `!` and hyphens stay.

---

## 📌 Section 4: Text Chunking (Conceptual Questions)

### Question 10: Why Split Text into Chunks? 💡

**Question**: Why does the `YouTubeSummarizer` class split large transcripts into chunks before summarizing them?

**A)** To make processing faster  
**B)** Because LLMs have token limits and long texts can exceed context length  
**C)** To make the summary shorter  
**D)** To improve summary quality  

**Hint**: 
- What is a "token"? Roughly 1 word = 1.3 tokens
- GPT-3.5 has ~4,000 token limit
- GPT-4 has ~8,000 token limit
- A 1-hour YouTube video transcript can be 50,000+ tokens!

Your Answer: _____ (select best answer)

In [ ]:
# Let's calculate token counts for context!

example_transcript = """
The modern world is built on technology. From smartphones to cloud computing,
technology shapes how we work, learn, and communicate. In this presentation,
we'll explore the future of AI and machine learning. These technologies are
becoming increasingly important in every industry. Let's dive into the details.
"""

# Rough estimation: 1 token ≈ 4 characters
token_estimate = len(example_transcript) / 4

print("Token estimation:")
print(f"  Transcript length: {len(example_transcript)} characters")
print(f"  Estimated tokens: ~{token_estimate:.0f} tokens")
print(f"\nLLM Token Limits:")
print(f"  GPT-3.5-turbo: 4,096 tokens")
print(f"  GPT-4: 8,192 tokens")
print(f"\nA 60-minute video transcript might have:")
print(f"  ~15,000-20,000 tokens")
print(f"  ❌ TOO LARGE for most models!")
print(f"\nSolution: Split into chunks!")
print(f"  Each chunk: ~1,000-2,000 tokens")
print(f"  ✅ Fits within context limit")

**EXPLANATION FOR Q10:**

The correct answer is **B** (and optionally **D**).

**Why chunk text?**

1. **Token Limits**: LLMs have maximum context windows
   - GPT-3.5: 4,096 tokens
   - GPT-4: 8,192 tokens
   - A 1-hour video: 15,000+ tokens → Doesn't fit!

2. **Quality**: Smaller, focused chunks lead to better summaries
   - Model can focus on specific sections
   - Less likely to miss important details

3. **Cost**: More tokens = more expensive
   - Chunking allows better cost management

4. **Processing**: Allows batch processing of multiple videos

Without chunking: API call fails with "token limit exceeded" ❌
With chunking: Works perfectly! ✅

---

## 📌 Section 5: Prompt Templates (Practical Questions)

### Question 11: Comparing Summarization Styles 🎯

**Question**: For each task, which summarization style would be BEST suited?

**Available styles**:
- **Concise**: 3-4 sentence summary (quick overview)
- **Detailed**: Overview + bullet points + conclusions (comprehensive)
- **Structured**: What/Why/How/Outcomes format (analytical)

**Match the task with the style:**

1. Create a Twitter post about a 20-minute tutorial video  
   → Best style: _____________

2. Write comprehensive study notes for an exam  
   → Best style: _____________

3. Analyze "What were the key takeaways and why they matter?"  
   → Best style: _____________

**Hint**: Think about the output format needed and level of detail.

In [ ]:
# Let's look at the different prompt templates

print("=" * 70)
print("SUMMARIZATION STYLES COMPARISON")
print("=" * 70)

styles = {
    "CONCISE": {
        "Output length": "3-4 sentences",
        "Best for": "Twitter, quick overviews, busy people",
        "Example": "This tutorial teaches Python basics in 20 minutes..."
    },
    "DETAILED": {
        "Output length": "Overview + 4-6 bullet points + conclusions",
        "Best for": "Study notes, documentation, thorough understanding",
        "Example": "Overview: ...\nKey Points:\n- Point 1\n- Point 2\nConclusions: ..."
    },
    "STRUCTURED": {
        "Output length": "What/Why/How/Outcomes format",
        "Best for": "Analysis, business decisions, research",
        "Example": "What: ...learned about Python\nWhy: ...important for development\nHow: ...through hands-on examples\nOutcomes: ..."
    }
}

for style, details in styles.items():
    print(f"\n{style}")
    print("-" * 70)
    for key, value in details.items():
        print(f"  {key}: {value}")

**EXPLANATION FOR Q11:**

**Matching answers:**

1. **Create a Twitter post** → **CONCISE**
   - ✅ Twitter has character limit
   - ✅ Need quick, punchy summary
   - ✅ 3-4 sentences perfect for tweet thread

2. **Write comprehensive study notes** → **DETAILED**
   - ✅ Need thorough coverage
   - ✅ Bullet points help organization
   - ✅ Need all key points for exam prep

3. **Analyze key takeaways & why they matter** → **STRUCTURED**
   - ✅ What/Why structure perfect for analysis
   - ✅ Explains reasoning behind takeaways
   - ✅ How/Outcomes show application

This teaches an important lesson: **Choose the right tool for the job!** 🎯

---

## 🌟 BONUS CHALLENGES (Optional - For Advanced Learners)

### Bonus Challenge 1: Custom Prompt Creation ✨

**Question**: Write your own prompt template for an "Educational Summary" that:
- Extracts learning outcomes
- Identifies prerequisites
- Lists 3-5 key concepts to study

**Task**: Create a PromptTemplate with:
- Appropriate input variables
- Clear instructions for the LLM
- Structured output format

**Hint**: Use the existing prompt templates in the notebook as examples!

In [ ]:
# Step 1: Create your custom prompt template

# TODO: Fill in the template below
educational_summary_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
# WRITE YOUR TEMPLATE HERE
# Include:
# 1. Clear instructions
# 2. Input placeholder {text}
# 3. Desired output structure

Text:
{text}

Educational Summary:
"""
)

# Step 2: Test your template (optional)
test_text = """
Machine Learning is a subset of Artificial Intelligence that enables systems
to learn from data without being explicitly programmed. There are three main types:
supervised learning, unsupervised learning, and reinforcement learning. 
Supervised learning uses labeled data to train models. Common algorithms include
linear regression and decision trees. Applications range from image recognition
to natural language processing.
"""

# If your prompt is working, this will show the formatted prompt
formatted = educational_summary_prompt.format(text=test_text)
print("Your formatted prompt:")
print("-" * 70)
print(formatted)

**HINT FOR BONUS 1:**

A good educational prompt should ask for:

```
Learning Outcomes: What should students be able to do?
Prerequisites: What knowledge is needed first?
Key Concepts: 3-5 main ideas to understand
Study Tips: How to approach this topic effectively
Challenges: Common misconceptions or difficult areas
```

Compare your template to the existing ones in the main notebook!

### Bonus Challenge 2: Real-World Application Strategy 🎬

**Scenario**: You want to summarize a 2-hour conference talk (about 20,000 tokens).

**Question**: Which approach would be better?

**Option A**: Use the entire transcript in one API call to ChatOpenAI

**Option B**: Split into chunks, summarize each chunk, then summarize the summaries

**Your choice**: Option _____ (A or B)

**Explain your answer with at least 2 reasons:**

Reason 1: _________________________________________________________________

Reason 2: _________________________________________________________________

**Hint**: Consider:
- API token limits
- API costs
- Summary quality
- Processing time

In [ ]:
# Let's analyze this problem with real numbers!

print("ANALYZING: 2-HOUR CONFERENCE TALK")
print("=" * 70)

# Estimation
transcript_tokens = 20000  # Rough estimate for 2-hour video
gpt_35_limit = 4096
gpt_4_limit = 8192

print(f"\nTranscript size: ~{transcript_tokens:,} tokens")
print(f"GPT-3.5 limit: {gpt_35_limit:,} tokens")
print(f"GPT-4 limit: {gpt_4_limit:,} tokens")

print(f"\n{'OPTION A: Single API Call':^70}")
print("-" * 70)
print(f"Will it work? {'❌ NO' if transcript_tokens > gpt_35_limit else '✅ YES'}")
print(f"Reason: {transcript_tokens:,} > {gpt_35_limit:,} tokens")
print(f"Cost: High (all tokens processed at once)")
print(f"Quality: May lose details due to length")

print(f"\n{'OPTION B: Chunked Approach':^70}")
print("-" * 70)
chunk_size = 2000
num_chunks = transcript_tokens // chunk_size
print(f"Number of chunks: ~{num_chunks}")
print(f"Tokens per chunk: ~{chunk_size:,}")
print(f"Will it work? ✅ YES (fits in all models)")
print(f"Cost: Potentially more (summary of summaries)")
print(f"Quality: Better focus on each section")

print(f"\n{'RECOMMENDATION':^70}")
print("-" * 70)
print("Option B is better for:")
print("  • Staying within token limits")
print("  • Focusing on specific sections")
print("  • Avoiding information loss")

**EXPLANATION FOR BONUS 2:**

**Better choice: Option B** ✅

**Reasons:**

1. **Token Limits**: 20,000 tokens exceeds most model limits
   - GPT-3.5: 4,096 tokens (doesn't fit)
   - Option A: ❌ Fails immediately
   - Option B: ✅ Works with chunking

2. **Quality**: Chunking allows focused summarization
   - Large context can be overwhelming for models
   - Chunk 1 summary: "Introduction and overview"
   - Chunk 2 summary: "Main points and discussion"
   - Final summary: "Combines all summaries"

3. **Cost**: Can be optimized
   - More API calls but smaller ones
   - Can use cheaper models for intermediate summaries

This is a common pattern in production systems! 🚀

### Bonus Challenge 3: Error Handling & Troubleshooting 🔧

**Scenario**: You're extracting transcripts and `extract_transcript()` returns `None` for a YouTube URL that looks valid.

**Question**: What are 3 troubleshooting steps a user could try?

**Step 1**: _________________________________________________________________

**Step 2**: _________________________________________________________________

**Step 3**: _________________________________________________________________

**Hint**: Think about:
- Video settings and privacy
- Network/connection issues
- Library version issues

In [ ]:
# Here's a troubleshooting checklist for debugging transcript extraction

troubleshooting_steps = {
    "Step 1: Verify Video Accessibility": [
        "✓ Check if video is public (not private/unlisted)",
        "✓ Verify video has English captions enabled",
        "✓ Try opening URL in browser to confirm it exists",
        "✓ Look for 'CC' caption button in YouTube player"
    ],
    "Step 2: Check Network & Environment": [
        "✓ Verify internet connection is working",
        "✓ Test with a different video known to have captions (e.g., TED talk)",
        "✓ Check that API keys are loaded: os.getenv('OPENAI_API_KEY')",
        "✓ Verify .env file is in the notebook directory"
    ],
    "Step 3: Update & Reinstall": [
        "✓ Update yt_dlp: pip install --upgrade yt_dlp",
        "✓ YouTube sometimes changes their API, yt_dlp needs updates",
        "✓ Check for error messages in console output",
        "✓ Try with a fresh Python kernel (Restart from kernel menu)"
    ],
    "Step 4: Debug the Issue": [
        "✓ Add print statements to see where it fails",
        "✓ Check yt_dlp logs for specific error messages",
        "✓ Try the same URL with standalone yt-dlp command",
        "✓ Ask in Discord/live session with exact error message"
    ]
}

print("TRANSCRIPT EXTRACTION TROUBLESHOOTING GUIDE")
print("=" * 70)
for step, actions in troubleshooting_steps.items():
    print(f"\n{step}")
    for action in actions:
        print(f"  {action}")

print("\n" + "=" * 70)
print("Most common issue: Video has no captions!")
print("Solution: Test with a TED talk or tutorial video known to have captions")

**EXPLANATION FOR BONUS 3:**

**Common troubleshooting steps:**

1. **Verify Video Accessibility**
   - Check if video is public (private videos won't work)
   - Ensure captions are enabled
   - Try opening the URL in a browser

2. **Check Network & Environment**
   - Verify internet connection works
   - Test with a known good video (e.g., TED Talk)
   - Confirm API keys are loaded

3. **Update Libraries**
   - YouTube API changes frequently
   - Update yt_dlp: `pip install --upgrade yt_dlp`
   - Sometimes requires kernel restart

4. **Debug Systematically**
   - Add print statements to track where it fails
   - Check console error messages
   - Try with multiple different URLs

**Pro tip**: Test with a TED talk first! They always have English captions. 🎯

---

## ✅ Homework Completion Checklist

Mark these off as you complete each section:

**Section 1: Setup & Imports**
- [ ] Q1: Environment variables answered
- [ ] Q2: .env vs os.environ compared
- [ ] Q3: LangChain advantages identified
  
**Section 2: Transcript Extraction**
- [ ] Q4: JSON3 parsing tested and verified
- [ ] Q5: Three failure reasons listed
- [ ] Q6: Configuration purpose explained

**Section 3: Text Preprocessing**
- [ ] Q7: Preprocessing example completed
- [ ] Q8: Regex pattern explained
- [ ] Q9: Removal patterns identified

**Section 4: Text Chunking**
- [ ] Q10: Chunking purpose explained

**Section 5: Prompt Templates**
- [ ] Q11: Summarization styles matched

**Bonus Challenges** (Optional)
- [ ] B1: Custom prompt template created
- [ ] B2: Real-world strategy analyzed
- [ ] B3: Troubleshooting steps documented

---

## 🎓 What You've Learned

After completing this homework, you should understand:

✅ **Security**: Why we use `.env` files for credentials  
✅ **Architecture**: How LangChain simplifies LLM applications  
✅ **Data Processing**: How to extract and parse YouTube captions  
✅ **Text Cleaning**: How to preprocess text for LLMs  
✅ **LLM Limits**: Why we need to chunk large texts  
✅ **Prompts**: How to design prompts for different use cases  
✅ **Debugging**: How to troubleshoot when things go wrong  

---

## 📞 Next Steps

1. **Review**: Go back through the main notebook for any unclear concepts
2. **Test**: Run your code cells to verify they work
3. **Explore**: Try modifying the prompts with different styles
4. **Practice**: Test transcript extraction with your favorite YouTube videos
5. **Submit**: Review your completed notebook

---

## 💬 Questions?

- 📖 Review the **YT-Transcript-Summarizer-Student.ipynb** for code examples
- 📋 Check the **README.md** for learning objectives
- 🎤 Ask during the live session Q&A
- 💬 Post in Discord/community channel

**Great work! You've completed Week 6 homework! 🎉**